# BEVFusion Teacher Caching + KD Training

**Dissertation: Knowledge-Distilled BEV Perception for ADAS**

This notebook runs in **two phases**:

**Phase 1 — Teacher caching** (requires Python 3.8 env + mmdet3d):
- Installs BEVFusion dependencies in a conda env
- Downloads `bevfusion-det.pth` pretrained weights
- Runs teacher forward pass over all nuScenes samples
- Saves outputs to `data/teacher_cache/{split}/{token}.pt`

**Phase 2 — Student KD training** (standard Python 3.x + PyTorch 2.x):
- Loads cached teacher outputs
- Trains student for 20 epochs with L_total = α·L_feature + β·L_logit + L_task
- Saves `kd_best.pth` to Google Drive

---
**Before starting:** Runtime → Change runtime type → GPU (T4 is fine)

**Estimated time:** Phase 1 ~45min | Phase 2 ~3-4hrs (20 epochs)

## 0. Mount Google Drive and check GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# ── Paths ──────────────────────────────────────────────
DRIVE_ROOT    = '/content/drive/MyDrive/dissertation-bev'
NUSCENES_PATH = f'{DRIVE_ROOT}/data/nuscenes'
CACHE_PATH    = f'{DRIVE_ROOT}/data/teacher_cache'
CKPT_PATH     = f'{DRIVE_ROOT}/checkpoints'

import os
os.makedirs(CACHE_PATH, exist_ok=True)
os.makedirs(CKPT_PATH,  exist_ok=True)
print(f'\nDrive root: {DRIVE_ROOT}')
print(f'nuScenes:   {NUSCENES_PATH}')
print(f'Cache:      {CACHE_PATH}')

# ── Check nuScenes data ────────────────────────────────
print('\n--- nuScenes check ---')
if os.path.exists(NUSCENES_PATH):
    for item in sorted(os.listdir(NUSCENES_PATH)):
        item_path = os.path.join(NUSCENES_PATH, item)
        if os.path.isdir(item_path):
            n_files = sum(len(f) for _, _, f in os.walk(item_path))
            print(f'  ✅ {item}/  ({n_files} files)')
        else:
            print(f'  📄 {item}')
    print('\n✅ nuScenes found — ready to proceed')
else:
    print(f'❌ nuScenes NOT found at: {NUSCENES_PATH}')
    print()
    print('  Fix: go to drive.google.com and upload your local')
    print('  data/nuscenes/ folder to:')
    print(f'  MyDrive/dissertation-bev/data/nuscenes/')
    print()
    print('  Expected folders inside: v1.0-mini/, samples/, sweeps/, maps/')

## 1. Clone your dissertation repo

In [ ]:
import os

# Clone your repo (replace with your actual GitHub URL)
REPO_URL = 'https://github.com/neomond/lightweight-bev-adas.git'  # ← update this

if not os.path.exists('/content/dissertation-bev'):
    !git clone {REPO_URL} /content/dissertation-bev
else:
    !cd /content/dissertation-bev && git pull

%cd /content/dissertation-bev

# Symlink nuScenes data from Drive (avoids copying 3.88GB)
os.makedirs('data', exist_ok=True)
if not os.path.exists('data/nuscenes'):
    os.symlink(NUSCENES_PATH, 'data/nuscenes')
    print('Symlinked nuScenes data')

# Install student model dependencies
!pip install -q nuscenes-devkit pyquaternion ultralytics tensorboard
print('Student dependencies installed')

## 2. Install BEVFusion environment

BEVFusion requires a specific set of dependencies that are incompatible
with the student training environment. We install them into a separate
conda environment and run the teacher caching from there.

This cell takes ~10 minutes on first run.

In [ ]:
# Install conda (Colab doesn't have it by default)
import os, sys

CONDA = '/content/miniconda/bin/conda'
PIP38 = '/content/miniconda/envs/bevfusion/bin/pip'
PY38  = '/content/miniconda/envs/bevfusion/bin/python'

if os.path.exists(PY38):
    print('✅ BEVFusion env already exists — skipping install')
else:
    print('Installing BEVFusion env (first time, ~10 min)...')

    # Install conda
    !wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh
    !bash Miniconda3-latest-Linux-x86_64.sh -b -p /content/miniconda

    sys.path.insert(0, '/content/miniconda/lib/python3.8/site-packages')

    # Create Python 3.8 environment
    !{CONDA} create -n bevfusion python=3.8 -y -q
    print('✅ Created bevfusion conda env')

print(f'PY38: {PY38}')

In [ ]:
# Install PyTorch 1.10 + CUDA 11.3 (required by BEVFusion)
if os.path.exists('/content/miniconda/envs/bevfusion/lib/python3.8/site-packages/torch'):
    print('✅ PyTorch 1.10 already installed — skipping')
else:
    print('Installing PyTorch 1.10 + mmcv (5-8 min)...')

    !{PIP38} install -q torch==1.10.2+cu113 torchvision==0.11.3+cu113 \
        -f https://download.pytorch.org/whl/torch_stable.html

    !{PIP38} install -q mmcv-full==1.4.0 \
        -f https://download.openmmlab.com/mmcv/dist/cu113/torch1.10.0/index.html

    print('✅ PyTorch 1.10 + mmcv installed')

In [ ]:
# Clone and install BEVFusion
if not os.path.exists('/content/bevfusion'):
    !git clone https://github.com/mit-han-lab/bevfusion.git /content/bevfusion

%cd /content/bevfusion
!{PIP38} install -q -e .
!{PIP38} install -q nuscenes-devkit pyquaternion torchpack

# Download pretrained checkpoint
os.makedirs('pretrained', exist_ok=True)
if not os.path.exists('pretrained/bevfusion-det.pth'):
    # The checkpoint download script from the BEVFusion repo
    !{PY38} tools/download_pretrained.py  
    # If the above fails, manually download from the BEVFusion README URL:
    # !gdown --id [GDRIVE_FILE_ID] -O pretrained/bevfusion-det.pth
    print('Downloaded bevfusion-det.pth')
else:
    print('Checkpoint already exists')

%cd /content/dissertation-bev

## 3. Write the BEVFusion caching script

This script runs inside the BEVFusion environment (Python 3.8 + mmdet3d)
and saves teacher outputs in the format expected by `CachedTeacherDataset`.

In [ ]:
# Write the teacher extraction script that runs in the BEVFusion env
cache_script = f'''
import sys, os, torch
from pathlib import Path
from tqdm import tqdm

sys.path.insert(0, '/content/bevfusion')

from mmcv import Config
from mmcv.runner import load_checkpoint
from mmdet3d.models import build_model
from mmdet3d.datasets import build_dataset, build_dataloader

DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CACHE_ROOT = Path('{CACHE_PATH}')
CKPT       = '/content/bevfusion/pretrained/bevfusion-det.pth'
CFG_PATH   = '/content/bevfusion/configs/nuscenes/det/transfusion/secfpn/camera+lidar/swint_v0p075/convfuser.yaml'
DATA_ROOT  = '{NUSCENES_PATH}'

def extract_fused_bev(model, batch):
    """Hook into BEVFusion to extract intermediate fused BEV features."""
    captured = {{}}

    def hook_fn(module, input, output):
        # BEVFusion fuser output: (B, 256, 128, 128)
        captured["fused_bev"] = output.detach().cpu()

    # Register hook on the fusion module (ConvFuser)
    hook = model.fuser.register_forward_hook(hook_fn)

    with torch.no_grad():
        result = model(return_loss=False, **batch)

    hook.remove()
    return captured.get("fused_bev"), result


def cache_split(model, split):
    cache_dir = CACHE_ROOT / split
    cache_dir.mkdir(parents=True, exist_ok=True)

    # Build BEVFusion-style dataset
    cfg = Config.fromfile(CFG_PATH)
    cfg.data_root = DATA_ROOT + "/"
    cfg.data[split].ann_file = f"{{DATA_ROOT}}/nuscenes_infos_{{split}}.pkl"
    cfg.data[split].data_root = DATA_ROOT + "/"

    dataset = build_dataset(cfg.data[split])
    loader  = build_dataloader(
        dataset, samples_per_gpu=1, workers_per_gpu=2,
        dist=False, shuffle=False
    )

    computed = 0
    for i, batch in enumerate(tqdm(loader, desc=f"Caching {{split}}")):
        # Get sample token from batch metadata
        token = batch["img_metas"][0].data[0][0]["sample_idx"]
        out_path = cache_dir / f"{{token}}.pt"

        if out_path.exists():
            continue

        # Move to GPU
        for k in batch:
            if hasattr(batch[k], "to"):
                batch[k] = batch[k].to(DEVICE)

        try:
            fused_bev, result = extract_fused_bev(model, batch)

            # Extract detection outputs
            # BEVFusion returns heatmap+regression via TransFusion head
            # Shape: heatmap (B, 10, 128, 128), regression (B, 8, 128, 128)
            heatmap    = model.pts_bbox_head.heatmap_cache.cpu()     # captured in forward
            regression = model.pts_bbox_head.regression_cache.cpu()  # captured in forward

            torch.save({{
                "fused_bev":    fused_bev[0],    # (256, 128, 128)
                "heatmap":      heatmap[0],      # (10,  128, 128)
                "regression":   regression[0],   # (8,   128, 128)
                "sample_token": token,
            }}, out_path)
            computed += 1

        except Exception as e:
            print(f"  Error on {{token}}: {{e}}")

    print(f"  {{split}}: {{computed}} samples cached to {{cache_dir}}")


# Load model
print("Loading BEVFusion...")
cfg = Config.fromfile(CFG_PATH)
model = build_model(cfg.model)
load_checkpoint(model, CKPT, map_location="cpu")
model = model.to(DEVICE).eval()
print(f"BEVFusion loaded on {{DEVICE}}")

for split in ["train", "val"]:
    cache_split(model, split)

print("\\n✅ All teacher outputs cached.")
print(f"Cache location: {{CACHE_ROOT}}")
'''

with open('/content/dissertation-bev/scripts/extract_teacher_bevfusion.py', 'w') as f:
    f.write(cache_script)

print('Wrote extract_teacher_bevfusion.py')

## 4. Run teacher caching (Phase 1)

This runs in the BEVFusion conda environment. Takes ~30-45 minutes for
the full nuScenes mini dataset. Outputs go directly to Google Drive.

In [ ]:
# Verify nuScenes info files exist (needed by BEVFusion's dataloader)
# If they don't exist, BEVFusion needs to generate them first
import os

info_train = f'{NUSCENES_PATH}/nuscenes_infos_train.pkl'
info_val   = f'{NUSCENES_PATH}/nuscenes_infos_val.pkl'

if not os.path.exists(info_train):
    print('INFO FILES MISSING — generating (takes ~5min)...')
    !{PY38} /content/bevfusion/tools/create_data.py nuscenes \
        --root-path {NUSCENES_PATH} \
        --out-dir {NUSCENES_PATH} \
        --extra-tag nuscenes
else:
    print(f'Info files found: {info_train}')

In [ ]:
# Run teacher caching in the BEVFusion Python 3.8 environment
!{PY38} /content/dissertation-bev/scripts/extract_teacher_bevfusion.py

In [ ]:
# Verify cache
import os
from pathlib import Path
import torch

for split in ['train', 'val']:
    files = list(Path(f'{CACHE_PATH}/{split}').glob('*.pt'))
    print(f'{split}: {len(files)} files cached')

# Check one file
sample = torch.load(files[0], weights_only=True)
print(f'\nSample file contents:')
for k, v in sample.items():
    if hasattr(v, 'shape'):
        print(f'  {k}: {v.shape}')
    else:
        print(f'  {k}: {v}')

## 5. Install student training dependencies (Phase 2)

Switch back to Colab's default Python for student training.
PyTorch 2.x + CUDA is already available in Colab.

In [ ]:
# Install student dependencies (in Colab's default Python env)
!pip install -q nuscenes-devkit pyquaternion ultralytics tensorboard

import torch
print(f'Student env: PyTorch {torch.__version__}, CUDA {torch.cuda.is_available()}')

# Symlink cache into repo
import os
os.makedirs('/content/dissertation-bev/data', exist_ok=True)
cache_link = '/content/dissertation-bev/data/teacher_cache'
if not os.path.exists(cache_link):
    os.symlink(CACHE_PATH, cache_link)
    print(f'Symlinked teacher cache: {cache_link} -> {CACHE_PATH}')

%cd /content/dissertation-bev

## 6. Run full KD training (Phase 2)

20 epochs, real teacher cache, full nuScenes mini.
Estimated time: ~3-4 hours on T4 GPU.

Checkpoints are saved to Google Drive every epoch.

In [ ]:
import subprocess, sys
os.makedirs(f'{CKPT_PATH}', exist_ok=True)

# Run KD training — checkpoints saved to Drive automatically
# (train_with_kd.py saves to checkpoints/ which we symlink to Drive below)
ckpt_link = '/content/dissertation-bev/checkpoints'
if not os.path.exists(ckpt_link):
    os.symlink(CKPT_PATH, ckpt_link)

logs_link = '/content/dissertation-bev/logs'
os.makedirs(f'{DRIVE_ROOT}/logs', exist_ok=True)
if not os.path.exists(logs_link):
    os.symlink(f'{DRIVE_ROOT}/logs', logs_link)

print('Starting KD training...')
print('Checkpoints → Google Drive:', CKPT_PATH)
print('Logs        → Google Drive:', f'{DRIVE_ROOT}/logs')

In [ ]:
# Full 20-epoch KD training with real cached teacher outputs
!python scripts/train_with_kd.py \
    --config configs/student.yaml \
    --epochs 20 \
    --batch 4 \
    --workers 4 \
    --run-name kd_real \
    --cache-dir data/teacher_cache

In [ ]:
# Also train baseline for 20 epochs (needed for fair comparison)
!python scripts/train_baseline.py \
    --config configs/student.yaml \
    --epochs 20 \
    --batch 4 \
    --workers 4 \
    --run-name baseline_20ep

## 7. Verify results and download

Checkpoints are already in Google Drive. This cell prints the final
numbers for your dissertation results table.

In [ ]:
import torch
from pathlib import Path

results = {}
for name in ['baseline_20ep_best', 'kd_real_best']:
    path = Path(CKPT_PATH) / f'{name}.pth'
    if path.exists():
        ckpt = torch.load(path, map_location='cpu', weights_only=False)
        results[name] = ckpt['val_loss']
        print(f'{name}: val_loss = {ckpt["val_loss"]:.4f}  (epoch {ckpt["epoch"]})')
    else:
        print(f'{name}: NOT FOUND at {path}')

if len(results) == 2:
    delta = results['baseline_20ep_best'] - results['kd_real_best']
    pct   = delta / results['baseline_20ep_best'] * 100
    print(f'\nKD improvement: {delta:.4f} ({pct:.1f}% reduction in val loss)')
    print(f'This is your primary dissertation result.')

## Troubleshooting

**BEVFusion install fails:**
The repo is archived but the code still works. Most common issue is mmcv
version mismatch — ensure mmcv-full==1.4.0 with CUDA 11.3.

**`nuscenes_infos_train.pkl` not found:**
BEVFusion uses pre-processed info files. Run cell 4's `create_data.py` step.
For nuScenes mini, this takes ~5 minutes.

**Hook doesn't capture fused_bev:**
The hook targets `model.fuser` (ConvFuser module). If BEVFusion's structure
differs, inspect with `print(model)` and update the hook target.

**Colab disconnects mid-training:**
Resume with: `--resume checkpoints/kd_real_latest.pth`
The latest checkpoint is saved every epoch to Drive.

**Out of VRAM on T4 (16GB):**
Reduce batch size: `--batch 2`. The teacher caching step uses batch=1
already so it won't OOM.